To begin, make a copy of this notebook to your own Google Drive: **File $\rightarrow$ Save a copy in Drive**.

### Submission Instructions:
1. **Download the notebook** (in the menubar, select File $\rightarrow$ Download .ipynb).
2. **Upload the downloaded notebook (.ipynb file) to your repository**.

Make sure you fill in any place that says `YOUR CODE HERE`, and all the cells include their outputs.

**Note:** To use a GPU, do the following: Runtime $\rightarrow$ Change runtime type $\rightarrow$ GPU.

In [ ]:
!pip install datasets
!pip install transformers
!pip install nltk
!pip install evaluate

In [ ]:
import nltk
nltk.download('punkt_tab')
## If you're using older version of nltk, you might need to use this line instead:
# nltk.download('punkt')

In [ ]:
import torch
import numpy as np

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

### Embeddings Setup

- **Direct Upload:** Click the **Files** icon on the left sidebar $\rightarrow$ **Upload** (temporary per session).
- **Google Drive via `gdown`:**


```python
import gdown
gdown.download('https://drive.google.com/uc?export=download&id=1TKNu5YltNk29orM0jm4gcYU3HKUygFSY', 'glove_vectors.npy', quiet=False)
gdown.download('https://drive.google.com/uc?export=download&id=14zqYRN_MFHwRCEBdzuWLhHLF6f-aDaqy', 'glove_vocab.json', quiet=False)
```

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("stanfordnlp/imdb")

In [ ]:
EMBEDDINGS_PATH = "glove_vectors.npy"
VOCAB_PATH = "glove_vocab.json"

In [ ]:
from nltk.tokenize import word_tokenize
import json

with open(VOCAB_PATH) as f:
  vocab = json.load(f)

unk_idx = vocab["unk"]

def tokenize_function(example):
  tokenized_sentences = [word_tokenize(text.lower()) for text in example["text"]]
  tokenized_idx = [[vocab.get(token, unk_idx) for token in tokens] for tokens in tokenized_sentences]

  return {"labels":example['label'],'input_ids':tokenized_idx}

In [ ]:
train_dataset = raw_datasets['train'].shuffle(seed=42).map(tokenize_function,batched=True)
eval_dataset = raw_datasets['test'].shuffle(seed=42).map(tokenize_function,batched=True)

In [12]:
def pad_sequence_to_length(
    sequence,
    desired_length: int,
    default_value = lambda: 0,
    padding_on_right: bool = True,
):
    sequence = list(sequence)
    # Truncates the sequence to the desired length.
    if padding_on_right:
        padded_sequence = sequence[:desired_length]
    else:
        padded_sequence = sequence[-desired_length:]
    # Continues to pad with default_value() until we reach the desired length.
    pad_length = desired_length - len(padded_sequence)
    # This just creates the default value once, so if it's a list, and if it gets mutated
    # later, it could cause subtle bugs. But the risk there is low, and this is much faster.
    values_to_pad = [default_value()] * pad_length
    if padding_on_right:
        padded_sequence = padded_sequence + values_to_pad
    else:
        padded_sequence = values_to_pad + padded_sequence
    return padded_sequence

In [ ]:
from evaluate import load
metric = load("accuracy")

### If you're using older versions, use:
# metric = dataset.load_metric("accuracy")

def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [15]:
from dataclasses import dataclass

@dataclass
class DataCollatorWithPadding:

  def __call__(self, features):
    features_dict={}

    # Collect labels into a single tensor if present
    if "labels" in features[0]:
      features_dict["labels"] = torch.tensor([x["labels"] for x in features], dtype=torch.long)

    # Extract token sequences and determine the batch-level max sequence length
    input_ids = [x["input_ids"] for x in features]
    max_len = max(len(x) for x in input_ids)

    # Dynamically pad sequences to max_len
    features_dict["input_ids"] = torch.tensor([pad_sequence_to_length(x, max_len) for x in input_ids], dtype=torch.long)

    # Length-based attention mask: 1 for real tokens, 0 for padding
    masks = [[1] * len(x) for x in input_ids]
    features_dict["attention_masks"] = torch.tensor([pad_sequence_to_length(x, max_len) for x in masks], dtype=torch.long)

    return features_dict


## SECTION A - DAN implementation

Implement `DAN` **once**. Add arguments to `__init__` so Sections B–D can reuse the same class.

In [ ]:
from torch import nn

# Use nn.Sequential and nn.Linear for the network, and nn.CrossEntropyLoss for the loss.
# Make sure that the final layer has output dimension of size 2.
class DAN(nn.Module):
  def __init__(self): # YOU ARE ALLOWED TO ADD ARGUMENTS TO THE __INIT__
          super().__init__()
          self.num_labels = 2
          # freeze=True keeps the pre-trained embeddings fixed during training (required).
          self.embeddings = nn.Embedding.from_pretrained(torch.FloatTensor(np.load(EMBEDDINGS_PATH)),
                                                         freeze=True)
          # YOUR CODE HERE

          # END YOUR CODE


  def forward(self, input_ids, attention_masks, labels=None, **kwargs):
      # YOUR CODE HERE
      avg = None
      # END YOUR CODE
      res = self.classifier(avg)
      loss = self.loss(res, labels)
      return {"loss": loss, "logits": res}


In [ ]:
# Define any TrainerCallback you need for the eval-accuracy-vs-epochs plot
# *before* creating the Trainer below.
# See: https://huggingface.co/transformers/main_classes/callback.html
from transformers import TrainerCallback

# YOUR CODE HERE

# END YOUR CODE

In [ ]:
from transformers import Trainer, TrainingArguments

co = DataCollatorWithPadding()
training_args = TrainingArguments("DAN",
                                  # YOUR CODE HERE
                                  num_train_epochs= , #must be at least 10.
                                  per_device_train_batch_size=,
                                  per_device_eval_batch_size=,
                                  learning_rate= ,
                                  # END YOUR CODE

                                  save_total_limit=2,
                                  log_level="error",
                                  eval_strategy="epoch")
model = DAN()



trainer = Trainer(
    model=model,
    data_collator=co,
    args=training_args,
    callbacks = [
                 # YOUR CODE HERE

                 # END YOUR CODE
    ],
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
# You may add Cells & Code lines if you need

## SECTION B - Trying dropout

> **Read Section 3.1 of the paper carefully before implementing.**  

In [ ]:
# YOUR CODE HERE

# END YOUR CODE

## SECTION C - Different Layer numbers

In [ ]:
# YOUR CODE HERE

# END YOUR CODE

## SECTION D - Activations

In [ ]:
# YOUR CODE HERE

# END YOUR CODE

## SECTION E - Sample 5 Misclassified Examples

Using your **top-performing** model from parts (a)–(d), find five movie reviews from the evaluation set that the model classified incorrectly. For each review, include a short explanation in your report describing why the model likely failed (See section 5.2 of the paper).

> `eval_dataset` was shuffled, so make sure you read the review text from the right source.

In [ ]:
# YOUR CODE HERE

# END YOUR CODE

# SECTION F — Your embeddings vs. GloVe

Use your own Q2 word2vec embeddings, using the same architecture and the given hyperparameters. Bring in `word2vec_vectors.npy` and `word2vec_vocab.json` the same way as the GloVe files in Embeddings Setup, then run the cells below in order.

In [ ]:
EMBEDDINGS_PATH = "word2vec_vectors.npy"
VOCAB_PATH = "word2vec_vocab.json"

with open(VOCAB_PATH) as f:
    vocab = json.load(f)

unk_idx = vocab["unk"]

train_dataset = raw_datasets["train"].shuffle(seed=42).map(tokenize_function, batched=True)
eval_dataset = raw_datasets["test"].shuffle(seed=42).map(tokenize_function, batched=True)

In [ ]:
# Train a new model.
# YOUR CODE HERE

# END YOUR CODE